# 面试问题：Pipeline Parallel 的 GPipe 与 1F1B 怎样调度，Bubble 和激活峰值如何计算？

可以直接复述的回答是：第一，把全局 batch 切成 microbatch 后送入多个 stage。第二，GPipe 先完成全部 forward 再 backward，容易理解但激活驻留高。第三，1F1B 在 warmup 后交替 forward/backward，优先释放旧激活。第四，每个操作必须满足跨 stage 的 forward 或 backward 依赖。第五，microbatch 太少时 pipeline bubble 占比很高。第六，要展示时间槽、空闲槽、makespan 和各 stage 激活峰值。下面用四 stage、六 microbatch 的训练任务模拟。

## 真实案例：四卡客服模型微调批次

一个全局 batch 包含 6 个脱敏客服 microbatch，每个给出 token 数和主要意图。四个逻辑 stage 分别代表 embedding、attention、MLP 和 loss。为突出调度依赖，每个操作占一个离散时间槽；真实 GPU 的 stage 时长和通信并不相等。

In [1]:
microbatches = [  # 定义六个具有真实训练语义的微批次
    {"id": 0, "tokens": 480, "topic": "退款"},  # 退款对话微批次
    {"id": 1, "tokens": 520, "topic": "物流"},  # 物流问答微批次
    {"id": 2, "tokens": 450, "topic": "账户"},  # 账户问题微批次
    {"id": 3, "tokens": 610, "topic": "合同"},  # 长合同咨询微批次
    {"id": 4, "tokens": 390, "topic": "发票"},  # 发票请求微批次
    {"id": 5, "tokens": 570, "topic": "故障"},  # 故障排查微批次
]  # 结束一个全局 batch
stages = ["embedding", "attention", "mlp", "loss"]  # 定义四个顺序模型 stage
print("训练输入：microbatch | tokens | topic")  # 展示调度器处理的真实批次字段
for microbatch in microbatches:  # 逐条输出六个微批次
    print(f"{microbatch['id']} | {microbatch['tokens']:3} | {microbatch['topic']}")  # 呈现长度差异和业务语义
print("Pipeline stages：", stages)  # 展示模型切分顺序


训练输入：microbatch | tokens | topic
0 | 480 | 退款
1 | 520 | 物流
2 | 450 | 账户
3 | 610 | 合同
4 | 390 | 发票
5 | 570 | 故障
Pipeline stages： ['embedding', 'attention', 'mlp', 'loss']


## Baseline / 基线：GPipe 全 Forward 后全 Backward

离散调度器每个 stage 每个时间槽最多执行一个操作。GPipe 在所有 forward 完成前不释放任何 backward，因此早期 stage 会积累更多激活。

In [2]:
def build_schedule(mode, microbatch_count, stage_count):  # 构建 GPipe 或 backward-priority 1F1B 离散时间表
    forward_done = set()  # 记录已完成的 forward 二元组
    backward_done = set()  # 记录已完成的 backward 二元组
    timeline = []  # 保存每个时间槽的各 stage 操作
    peak_activations = [0] * stage_count  # 记录各 stage 尚未反传的激活峰值
    target = microbatch_count * stage_count  # 计算 forward 或 backward 总操作数
    while len(backward_done) < target:  # 持续调度直到所有反向操作完成
        decisions = [None] * stage_count  # 初始化当前时间槽四个 stage 的操作
        all_forward_complete = len(forward_done) == target  # GPipe 只有全部 forward 完成后才允许 backward
        for stage in range(stage_count):  # 为每个 stage 选择一个依赖就绪操作
            ready_backward = [microbatch for microbatch in range(microbatch_count) if (stage, microbatch) in forward_done and (stage, microbatch) not in backward_done and (stage == stage_count - 1 or (stage + 1, microbatch) in backward_done)]  # 查找梯度依赖就绪的反向操作
            ready_forward = [microbatch for microbatch in range(microbatch_count) if (stage, microbatch) not in forward_done and (stage == 0 or (stage - 1, microbatch) in forward_done)]  # 查找激活依赖就绪的前向操作
            allow_backward = mode == "1f1b" or all_forward_complete  # 1F1B 可尽早反传，GPipe 需等待全部前向
            if allow_backward and ready_backward:  # 优先反传最早的可用微批次以释放激活
                decisions[stage] = ("B", min(ready_backward))  # 为当前 stage 选择一个 backward
            elif ready_forward and not (mode == "gpipe" and all_forward_complete):  # 仍处于前向阶段时选择最早微批次
                decisions[stage] = ("F", min(ready_forward))  # 为当前 stage 选择一个 forward
        timeline.append(decisions)  # 保存当前时间槽的并行操作
        for stage, decision in enumerate(decisions):  # 在时间槽末统一提交完成状态
            if decision is None:  # 空闲 stage 不更新依赖
                continue  # 保留 bubble 供后续统计
            direction, microbatch = decision  # 读取操作方向和微批次编号
            if direction == "F":  # 前向完成后激活可供下一 stage 和本 stage 反传
                forward_done.add((stage, microbatch))  # 标记当前前向依赖完成
            else:  # 反向完成后梯度可传给前一 stage
                backward_done.add((stage, microbatch))  # 标记当前反向依赖完成
        for stage in range(stage_count):  # 更新时间槽末各 stage 驻留激活数
            live = sum((stage, microbatch) in forward_done and (stage, microbatch) not in backward_done for microbatch in range(microbatch_count))  # 统计已前向但未反向的微批次
            peak_activations[stage] = max(peak_activations[stage], live)  # 更新当前 stage 的激活峰值
    return timeline, peak_activations  # 返回完整时间表和各 stage 激活峰值
gpipe_timeline, gpipe_peak = build_schedule("gpipe", len(microbatches), len(stages))  # 构建全前向后全反向基线
print("GPipe 前 14 个时间槽：t | " + " | ".join(stages))  # 输出能够观察填充和排空的时间表
for time, slot in enumerate(gpipe_timeline[:14]):  # 展示前十四个关键时间槽
    labels = ["-" if item is None else f"{item[0]}{item[1]}" for item in slot]  # 把操作转换为紧凑 F/B 标签
    print(f"{time:2} | " + " | ".join(f"{label:9}" for label in labels))  # 逐槽展示四 stage 并行状态
print("GPipe 激活峰值：", dict(zip(stages, gpipe_peak)))  # 展示全前向策略的内存压力


GPipe 前 14 个时间槽：t | embedding | attention | mlp | loss
 0 | F0        | -         | -         | -        
 1 | F1        | F0        | -         | -        
 2 | F2        | F1        | F0        | -        
 3 | F3        | F2        | F1        | F0       
 4 | F4        | F3        | F2        | F1       
 5 | F5        | F4        | F3        | F2       
 6 | -         | F5        | F4        | F3       
 7 | -         | -         | F5        | F4       
 8 | -         | -         | -         | F5       
 9 | -         | -         | -         | B0       
10 | -         | -         | B0        | B1       
11 | -         | B0        | B1        | B2       
12 | B0        | B1        | B2        | B3       
13 | B1        | B2        | B3        | B4       
GPipe 激活峰值： {'embedding': 6, 'attention': 6, 'mlp': 6, 'loss': 6}


## 核心实现：Backward-ready 优先的 1F1B

同一依赖引擎改为在 backward 就绪时优先执行，从最后 stage 开始尽早释放已完成 forward 的 microbatch。

In [3]:
onef_timeline, onef_peak = build_schedule("1f1b", len(microbatches), len(stages))  # 构建依赖正确的 backward-priority 1F1B 时间表
print("1F1B 前 14 个时间槽：t | " + " | ".join(stages))  # 输出与 GPipe 同口径的时间槽
for time, slot in enumerate(onef_timeline[:14]):  # 展示 warmup 后的前反向交替
    labels = ["-" if item is None else f"{item[0]}{item[1]}" for item in slot]  # 把调度决策转换为 F/B 标签
    print(f"{time:2} | " + " | ".join(f"{label:9}" for label in labels))  # 观察哪些 stage 在同一时间处理不同方向
print("1F1B 激活峰值：", dict(zip(stages, onef_peak)))  # 展示尽早反传后的峰值变化


1F1B 前 14 个时间槽：t | embedding | attention | mlp | loss
 0 | F0        | -         | -         | -        
 1 | F1        | F0        | -         | -        
 2 | F2        | F1        | F0        | -        
 3 | F3        | F2        | F1        | F0       
 4 | F4        | F3        | F2        | B0       
 5 | F5        | F4        | B0        | F1       
 6 | -         | B0        | F3        | B1       
 7 | B0        | F5        | B1        | F2       
 8 | -         | B1        | F4        | B2       
 9 | B1        | -         | B2        | F3       
10 | -         | B2        | F5        | B3       
11 | B2        | -         | B3        | F4       
12 | -         | B3        | -         | B4       
13 | B3        | -         | B4        | F5       
1F1B 激活峰值： {'embedding': 6, 'attention': 5, 'mlp': 3, 'loss': 1}


## 失败案例与修正：各 stage 本地交替会违反跨 stage 依赖

若每个 stage 都机械执行 F0、B0、F1、B1，最后 stage 会在尚未收到 F0 激活时尝试 B0。修正必须由全局依赖就绪条件释放操作，而不是按局部字符串轮换。

In [4]:
naive_slots = [[("F", 0), ("F", 0), ("F", 0), ("B", 0)], [("B", 0), ("B", 0), ("B", 0), ("F", 0)]]  # 构造局部交替导致的两槽错误计划
def dependency_errors(timeline, stage_count):  # 检查任意时间表是否满足前向和反向依赖
    forward_done = set()  # 保存此前时间槽完成的前向操作
    backward_done = set()  # 保存此前时间槽完成的反向操作
    errors = []  # 收集违规操作和原因
    for time, slot in enumerate(timeline):  # 按时间顺序验证计划
        for stage, decision in enumerate(slot):  # 检查当前时间槽的每个 stage
            if decision is None:  # 空闲槽没有依赖风险
                continue  # 跳过 bubble
            direction, microbatch = decision  # 读取操作身份
            if direction == "F" and stage > 0 and (stage - 1, microbatch) not in forward_done:  # 前向需要上一 stage 在更早时间完成
                errors.append((time, stage, direction, microbatch, "missing_forward_input"))  # 记录缺失激活
            if direction == "B" and ((stage, microbatch) not in forward_done or (stage < stage_count - 1 and (stage + 1, microbatch) not in backward_done)):  # 反向同时需要本 stage 激活与下一 stage 梯度
                errors.append((time, stage, direction, microbatch, "missing_backward_dependency"))  # 记录缺失梯度或激活
        for stage, decision in enumerate(slot):  # 时间槽末提交操作完成状态
            if decision is not None and decision[0] == "F":  # 提交当前前向完成
                forward_done.add((stage, decision[1]))  # 为后续时间槽开放依赖
            elif decision is not None:  # 提交当前反向完成
                backward_done.add((stage, decision[1]))  # 为前一 stage 开放梯度
    return errors  # 返回所有可定位的依赖违规
naive_errors = dependency_errors(naive_slots, len(stages))  # 验证局部交替反例
onef_errors = dependency_errors(onef_timeline, len(stages))  # 验证依赖驱动 1F1B 时间表
print("局部交替错误：", naive_errors)  # 展示错误发生的时间槽、stage 和依赖
print("依赖驱动 1F1B 错误数：", len(onef_errors))  # 展示修正时间表没有非法操作


局部交替错误： [(0, 1, 'F', 0, 'missing_forward_input'), (0, 2, 'F', 0, 'missing_forward_input'), (0, 3, 'B', 0, 'missing_backward_dependency'), (1, 0, 'B', 0, 'missing_backward_dependency'), (1, 1, 'B', 0, 'missing_backward_dependency')]
依赖驱动 1F1B 错误数： 0


## 结果表：Makespan、Bubble 与激活峰值

In [5]:
def schedule_metrics(timeline, peaks):  # 计算离散调度的吞吐和内存代理指标
    total_slots = len(timeline) * len(stages)  # 计算所有 stage 可用时间槽数量
    busy_slots = sum(item is not None for slot in timeline for item in slot)  # 统计真正执行前向或反向的槽位
    bubble_ratio = 1 - busy_slots / total_slots  # 计算空闲槽比例
    return len(timeline), bubble_ratio, max(peaks), sum(peaks)  # 返回总时间、bubble 和激活峰值指标
gpipe_metrics = schedule_metrics(gpipe_timeline, gpipe_peak)  # 计算 GPipe 指标
onef_metrics = schedule_metrics(onef_timeline, onef_peak)  # 计算 1F1B 指标
print("schedule | makespan_slots | bubble_ratio | max_live_activations | sum_stage_peaks")  # 输出同一依赖图下的调度对照
print(f"gpipe | {gpipe_metrics[0]:3} | {gpipe_metrics[1]:.1%} | {gpipe_metrics[2]} | {gpipe_metrics[3]}")  # 展示全前向基线
print(f"1f1b | {onef_metrics[0]:3} | {onef_metrics[1]:.1%} | {onef_metrics[2]} | {onef_metrics[3]}")  # 展示尽早反传方案
small_timeline, small_peak = build_schedule("1f1b", 2, len(stages))  # 构造 microbatch 少于 stage 的低利用率边界
small_metrics = schedule_metrics(small_timeline, small_peak)  # 计算小批次 bubble 比例
print(f"仅 2 个 microbatch 时：makespan={small_metrics[0]}，bubble={small_metrics[1]:.1%}")  # 展示微批数量不足的失败场景


schedule | makespan_slots | bubble_ratio | max_live_activations | sum_stage_peaks
gpipe |  18 | 33.3% | 6 | 24
1f1b |  18 | 33.3% | 6 | 15
仅 2 个 microbatch 时：makespan=10，bubble=60.0%


## 结果解读

GPipe 在 backward 开始前让早期 stage 保存多个未反传激活；1F1B 在依赖就绪后优先反传，降低激活峰值。两者执行相同数量的前反向操作，离散模型中的 makespan 和 bubble 取决于调度优先级。只有两个 microbatch 时，四 stage 很难填满，说明并行度不能脱离 microbatch 数量选择。

## 生产边界

真实调度要考虑各 stage FLOPs 不平衡、通信时延、activation checkpoint、梯度累积、虚拟 pipeline stage 和异步故障。1F1B 还需明确权重版本，避免跨 update 使用不同参数。本例所有操作等长，没有模拟 CUDA stream 和跨节点链路。

## 最小回归测试

In [6]:
assert len(microbatches) >= 5  # 保证训练案例包含足够多的真实微批次
assert len(naive_errors) > 0  # 保证局部交替反例确实违反跨 stage 依赖
assert onef_errors == []  # 保证依赖驱动 1F1B 时间表所有操作合法
assert onef_metrics[2] <= gpipe_metrics[2]  # 保证尽早反传不会增加最大激活驻留数
assert sum(item is not None for slot in onef_timeline for item in slot) == len(microbatches) * len(stages) * 2  # 保证每个微批在每个 stage 恰有一次前向和反向
assert small_metrics[1] > 0  # 保证微批过少时可以观察到 pipeline bubble
